In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import numpy as np
from tqdm import tqdm  
import time
from sklearn.preprocessing import StandardScaler
import pandas as pd

In [ ]:
class StockDataset_by_year(Dataset):
    # task任务类型，默认为回归
    def __init__(self, base_dir, start_year, end_year, task='regression'):
        self.base_dir = base_dir
        self.years = list(range(start_year, end_year + 1))
        self.task = task

        self.data_full = []
        self.features_list = []
        self.labels_list = []

        # 加载数据并使用进度条
        for year in tqdm(self.years, desc="Loading Data by Year"):
            data_file = os.path.join(base_dir, f"{year}_df_data.feather")
            label_file = os.path.join(base_dir, f"{year}_real_RET.feather")

            # 读取数据和标签
            X = pd.read_feather(data_file)
            y = pd.read_feather(label_file)
            print(f"Loaded data for year {year}:")

            # 如果任务是分类，则将y的值转化为0或1
            if self.task == 'classification':
                y['real_RET'] = (y['real_RET'] > 0).astype(float)

            combined_df = pd.concat([X, y['real_RET']], axis=1)

            reserved_cols = ['permno', 'DATE', 'mvel1', 'real_RET']
            feature_cols = [col for col in X.columns if col not in ['permno', 'DATE']]

            # 避免重复列
            all_needed_cols = reserved_cols + [col for col in feature_cols if col not in ['permno', 'DATE']]

            # 构造完整数据
            self.data_full.append(combined_df[all_needed_cols])

            self.features_list.append(combined_df[feature_cols].values)
            self.labels_list.append(combined_df['real_RET'].values)

        self.data_full = pd.concat(self.data_full, ignore_index=True)
        features_all = np.vstack(self.features_list)
        labels_all = np.hstack(self.labels_list)

        self.features = torch.tensor(features_all, dtype=torch.float32)
        self.labels = torch.tensor(labels_all, dtype=torch.float32)

        print(f"Data loaded from {start_year} to {end_year}.")
        print(f"Total samples: {len(self.labels)}")
        print(f"Feature dimension: {self.features.shape[1]}")
        print(f"Task: {self.task}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        X = self.features[idx]
        y = self.labels[idx]
        return X, y, idx
    


# 构建数据集和数据加载器的函数
def create_dataloaders(base_dir, start_year_train, start_year_test, end_year_test, task='regression', batch_size=32):
    # 构建训练集
    print("Constructing training dataset...")
    train_dataset = StockDataset_by_year(base_dir, start_year_train, start_year_test - 1, task)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # 构建测试集
    print("Constructing testing dataset...")
    test_dataset = StockDataset_by_year(base_dir, start_year_test, end_year_test, task)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from tqdm import tqdm
import pandas as pd
import os
import gc
import shutil
import time
# 检查 GPU 可用性
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# 计算R方（决定系数），R方度量了模型预测值和真实值之间的拟合程度，传参是真实值和预测值。
def calculate_r2_oos(real, predict):
    real = np.array(real)
    predict = np.array(predict)
    assert real.shape == predict.shape 
    valid_mask = ~np.isnan(real)
    numerator = np.sum((real[valid_mask] - predict[valid_mask]) ** 2)
    denominator = np.sum(real[valid_mask] ** 2)
    R2_oos = 1 - (numerator / denominator)
    return R2_oos

In [ ]:
def train_model_once(model, train_loader, test_loader, input_dim, num_epochs=50, lr=0.001, patience=10,
                     save_base_path=None, use_scheduler=False, run_idx=None):
    # 传参包括：train_loader:用于训练数据的Dataloader加载器、test_loader同理、num_epochs训练的总轮数、lr:学习率、patience：
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.MSELoss()  # 定义损失函数，这里使用均方误差，常用于回归任务
    optimizer = optim.Adam(model.parameters(), lr=lr)  # 使用Adam优化器来优化模型参数，学习率为lr

    # Optionally add a learning rate scheduler
    scheduler = None
    if use_scheduler:
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    results = {'train_loss': [], 'test_loss': [], 'test_r2': [], 'learning_rate': []}

    # 创建模型文件夹
    model_folder = os.path.join(save_base_path, model.__class__.__name__)
    if not os.path.exists(model_folder):
        os.makedirs(model_folder)

    total_start_time = time.time()

    # 添加进度条来跟踪epoch进度
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0

        # 在训练数据加载器上添加进度条
        train_loader_iter = tqdm(train_loader, desc=f"Epoch {epoch + 1} Training")
        for X_batch, y_batch in train_loader_iter:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)

        train_loss /= len(train_loader.dataset)
        results['train_loss'].append(train_loss)

        # Testing
        model.eval()
        test_loss = 0.0
        y_test_pred_list = []
        y_test_list = []
        # 在测试数据加载器上添加进度条
        test_loader_iter = tqdm(test_loader, desc=f"Epoch {epoch + 1} Testing")
        with torch.no_grad():
            for X_batch, y_batch in test_loader_iter:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                test_loss += loss.item() * X_batch.size(0)
                y_test_pred_list.append(outputs.cpu().numpy())
                y_test_list.append(y_batch.cpu().numpy())
        test_loss /= len(test_loader.dataset)
        y_test_pred = np.concatenate(y_test_pred_list).ravel()
        y_test = np.concatenate(y_test_list).ravel()
        test_r2 = calculate_r2_oos(y_test, y_test_pred)

        results['test_loss'].append(test_loss)
        results['test_r2'].append(test_r2)

        # Log the current learning rate if scheduler is used
        if scheduler:
            current_lr = scheduler.optimizer.param_groups[0]['lr']
            results['learning_rate'].append(current_lr)
            
        # Use scheduler if applicable
        if scheduler:
            scheduler.step(test_loss)

        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            best_model_state = model.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    total_training_time = time.time() - total_start_time

    # 保存最佳模型和历史记录
    r2_str = f"{test_r2:.6f}"
    file_suffix = f"NN2_Run:{run_idx}_Epochs:{num_epochs}_LR:{lr}_Patience:{patience}_R2_:{r2_str}"
    model_save_path = os.path.join(save_base_path, f"{file_suffix}.pth")
    history_save_path = os.path.join(save_base_path, f"{file_suffix}_history.txt")

    if best_model_state:
        model.load_state_dict(best_model_state)
        torch.save(model.state_dict(), model_save_path)

        with open(history_save_path, 'w') as f:
            f.write("Epoch\tTrain Loss\tTest Loss\tTest R²\tLR\n")
            for epoch in range(len(results['train_loss'])):
                f.write(
                    f"{epoch + 1}\t{results['train_loss'][epoch]:.6f}\t{results['test_loss'][epoch]:.6f}\t{results['test_r2'][epoch]:.6f}\t{results['learning_rate'][epoch] if use_scheduler else lr:.6f}\n")

        return model_save_path, results['test_r2'][-1], total_training_time
    else:
        return None, results['test_r2'][-1], total_training_time

In [ ]:
# 使用函数创建数据加载器
data_dir = 'data/processed/yearly'
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016
train_loader_reg_one_loop, test_loader_reg_one_loop = create_dataloaders(data_dir, start_year_train, start_year_test, end_year_test, task='regression')

In [ ]:
input_dim = train_loader_reg_one_loop.dataset[0][0].shape[0]
base_save_path = 'outputs'  # Kaggle 默认的工作目录
num_runs = 5
print(input_dim)

## 验证集预测

In [ ]:
class ModelRunnerByYear:
    """
    一个可复用的类，使用以加载的模型对验证集进行预测，将预测结果输出为CSV
    可将任意PyTorch模型实例传入run()方法进行训练和验证集预测。
    """

    def __init__(self,
                 base_dir,
                 val_start_year=1987,
                 val_end_year=2016,
                 output_csv="predictions_only.csv",
                 batch_size=64,
                 task='regression'):
        self.base_dir = base_dir
        self.val_start_year = val_start_year
        self.val_end_year = val_end_year
        self.output_csv = output_csv
        self.batch_size = batch_size
        self.task = task

        print('Loading validation dataset...')
        self.val_dataset = StockDataset_by_year(self.base_dir, self.val_start_year, self.val_end_year, task=self.task)
        self.val_dataloader = DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=True)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f'Prediction device:{self.device}')

    def run(self, model):
        model.to(self.device)
        model.eval()

        predictions = []
        indices = []
        with torch.no_grad():
            for X_batch, _, idx_batch in tqdm(self.val_dataloader, desc="Valid"):
                X_batch = X_batch.to(self.device)
                preds = model(X_batch).squeeze().cpu().numpy()
                predictions.append(preds)
                indices.extend(idx_batch.cpu().numpy())

        predictions = np.concatenate(predictions, axis=0)

        val_data_full = self.val_dataset.data_full.iloc[indices].copy()

        val_data_full.reset_index(drop=True, inplace=True)

        val_data_full['up_prob'] = predictions
        # 去除重复列
        val_data_full = val_data_full.loc[:, ~val_data_full.columns.duplicated()].copy()

        final_df = val_data_full[[ 'permno', 'DATE', 'up_prob', 'mvel1','real_RET']].copy()
        final_df.rename(columns={
            'permno': 'StockID',
            'DATE': 'ending_date',
            'mvel1': 'MarketCap',
            'real_RET': 'ret_val'
        }, inplace=True)
        
        # 确保列顺序
        final_df = final_df[['StockID', 'ending_date', 'up_prob', 'MarketCap', 'ret_val']]

        final_df.to_csv(self.output_csv, index=False)
        print(f"预测结果已保存至 {self.output_csv}，格式如下：")
        print(final_df.head())

In [ ]:
import torch
import torch.nn as nn

# 定义 NN2 模型
class NN2(nn.Module):
    def __init__(self, input_dim, hidden_neurons=128, dropout_rate=0.5):
        super(NN2, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_neurons)
        self.bn1 = nn.BatchNorm1d(hidden_neurons)
        self.fc2 = nn.Linear(hidden_neurons, hidden_neurons // 2)
        self.bn2 = nn.BatchNorm1d(hidden_neurons // 2)
        self.out = nn.Linear(hidden_neurons // 2, 1)
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        return self.out(x)

# 使用示例

if __name__ == "__main__":
    model = NN2(input_dim, hidden_neurons=128)

    # 加载事先训练好的最佳模型权重
    model.load_state_dict(torch.load('data/resources/one_loop_best_model.pth'))

    predictor = ModelRunnerByYear(
        base_dir='data/processed/yearly',
        val_start_year=1987,
        val_end_year=2016,
        output_csv="NN2_best_predictions.csv",
        batch_size=64,
        task='regression'
    )

    predictor.run(model)

## 模型性能评估

In [ ]:
# 展示ret_val和up_prob的分布图像
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取数据
file_path = 'outputs/NN2_3_best_predictions.csv'
data = pd.read_csv(file_path)

plt.figure(figsize=(60, 6))

# 绘制 ret_val 和 up_prob 的分布图
plt.hist(data['ret_val'], bins=1000, alpha=0.5, color='green', label='ret_val Distribution')
plt.hist(data['up_prob'], bins=50, alpha=0.5, color='blue', label='up_prob Distribution')

x_min, x_max = data[['up_prob', 'ret_val']].min().min(), data[['up_prob', 'ret_val']].max().max()
plt.xticks(np.linspace(x_min, x_max, 40))  # 将横轴分成 40 段

# 添加标签和标题
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Comparison of Distributions: up_prob on Top of ret_val')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)

# 保存图像到文件
save_path = 'outputs/NN3_distribution_plot.png'  # 可以改成其他路径和文件名
plt.savefig(save_path, dpi=300, bbox_inches='tight')

# 显示图像
plt.show()

# print(f"图像已保存到: {save_path}")
    

# 计算均方误差
def calculate_mse(y_true, y_pred):
    
    # 计算误差（真实值与预测值之差）
    error = y_true - y_pred
    # 计算均方误差
    mse = np.mean(error**2)
    return mse

MSE = calculate_mse(data.ret_val,data.up_prob)
print("MSE:",MSE)

# 计算R²
final_r2 = calculate_r2_oos(data.ret_val,data.up_prob)
print(f"验证集 R² 值: {final_r2:.6f}")

# 计算准确率
def convert_to_binary(value):
    return (1, 0) if value > 0 else (0, 1)
    
df = pd.read_csv('data/resources/NN1_best_predictions.csv')

df['up_prob_binary'] = df['up_prob'].apply(convert_to_binary)
df['ret_val_binary'] = df['ret_val'].apply(convert_to_binary)

correct_predictions = (df['up_prob_binary'] == df['ret_val_binary']).sum()
accuracy = correct_predictions / len(df)

print(f'准确率: {accuracy:.4f}')

## 计算投资组合

In [ ]:
import os
import os.path as op
import pdb

import numpy as np
import pandas as pd


class PortfolioManager(object):
    def __init__(
            self,
            signal_df: pd.DataFrame,
            freq: str,
            portfolio_dir: str,
            start_year=1987,
            end_year=2016,
            delay_list=None,  # 默认为 None，后续会自动设为 [0]
            load_signal=True,
            custom_ret=None,
            transaction_cost=False,
    ):
        assert freq in ["week", "month", "quarter"]
        self.freq = freq
        self.portfolio_dir = portfolio_dir
        self.start_year = start_year
        self.end_year = end_year
        self.no_delay_ret_name = f"next_{freq}_ret"
        self.custom_ret = custom_ret
        self.delay_list = [0] if delay_list is None else delay_list  # 默认为 [0]
        self.transaction_cost = transaction_cost
        if load_signal:
            assert "up_prob" in signal_df.columns
            self.signal_df = self.get_up_prob_with_period_ret(signal_df)

    def get_up_prob_with_period_ret(self, signal_df):
        # 直接使用输入的 signal_df
        signal_df = signal_df[
            signal_df.index.get_level_values("ending_date").year.isin(
                range(self.start_year, self.end_year + 1)
            )
        ]

        # 保留 signal_df 中已有的 MarketCap 和 ret_val 列
        signal_df.MarketCap = signal_df.MarketCap.abs()
        signal_df = signal_df[~signal_df.MarketCap.isnull()].copy()

        # 无需再添加 ret_val 这类列
        signal_df[self.no_delay_ret_name] = signal_df["ret_val"]
        return signal_df


    def calculate_portfolio_rets(self, weight_type, cut=10, delay=0):
        assert weight_type in ["ew", "vw"]
        assert delay == 0  # 只支持 delay = 0
        if self.custom_ret:
            print(f"Calculating portfolio using {self.custom_ret}")
            ret_name = self.custom_ret
        else:
            ret_name = self.no_delay_ret_name

        df = self.signal_df.copy()

        def __get_decile_df_with_inv_ret(reb_df, decile_idx):
            rebalance_up_prob = reb_df.up_prob
            low = np.percentile(rebalance_up_prob, decile_idx * 100.0 / cut)
            high = np.percentile(rebalance_up_prob, (decile_idx + 1) * 100.0 / cut)
            if decile_idx == 0:
                pf_filter = (rebalance_up_prob >= low) & (rebalance_up_prob <= high)
            else:
                pf_filter = (rebalance_up_prob > low) & (rebalance_up_prob <= high)
            _decile_df = reb_df[pf_filter].copy()
            if weight_type == "ew":
                stock_num = len(_decile_df)
                _decile_df["weight"] = (
                        pd.Series(
                            np.ones(stock_num), dtype=np.float64, index=_decile_df.index
                        )
                        / stock_num
                )
            else:
                value = _decile_df.MarketCap
                _decile_df["weight"] = pd.Series(value, dtype=np.float64) / np.sum(value)
            _decile_df["inv_ret"] = _decile_df["weight"] * _decile_df[ret_name]
            return _decile_df

        dates = np.sort(np.unique(df.index.get_level_values("ending_date")))
        print(f"Calculating portfolio from {dates[0]}, {dates[1]} to {dates[-1]}")
        turnover = np.zeros(len(dates) - 1)
        portfolio_ret = pd.DataFrame(index=dates, columns=list(range(cut)))
        prev_to_df = None
        #prob_ret_corr = []
        #prob_ret_pearson_corr = []
        prob_inv_ret_corr = []
        prob_inv_ret_pearson_corr = []

        for i, date in enumerate(dates):
            rebalance_df = df.loc[date].copy()
            rank_corr = ut.rank_corr(
                rebalance_df, "up_prob", ret_name, method="spearman"
            )
            pearson_corr = ut.rank_corr(
                rebalance_df, "up_prob", ret_name, method="pearson"
            )
            prob_ret_corr.append(rank_corr)
            prob_ret_pearson_corr.append(pearson_corr)

            low = np.percentile(rebalance_df.up_prob, 10)
            high = np.percentile(rebalance_df.up_prob, 90)
            if low == high:
                print(f"Skipping {date}")
                continue
            for j in range(cut):
                decile_df = __get_decile_df_with_inv_ret(rebalance_df, j)
                if self.transaction_cost:
                    if j == cut - 1:
                        decile_df["inv_ret"] -= (
                                decile_df["weight"] * decile_df["transaction_fee"] * 2
                        )
                    elif j == 0:
                        decile_df["inv_ret"] += (
                                decile_df["weight"] * decile_df["transaction_fee"] * 2
                        )
                portfolio_ret.loc[date, j] = np.sum(decile_df["inv_ret"])

            sell_decile = __get_decile_df_with_inv_ret(rebalance_df, 0)
            buy_decile = __get_decile_df_with_inv_ret(rebalance_df, cut - 1)
            buy_sell_decile = pd.concat([sell_decile, buy_decile])

            prob_inv_ret_corr.append(
                ut.rank_corr(buy_sell_decile, "up_prob", "inv_ret", method="spearman")
            )
            prob_inv_ret_pearson_corr.append(
                ut.rank_corr(buy_sell_decile, "up_prob", "inv_ret", method="pearson")
            )

            sell_decile[["weight", "inv_ret"]] = sell_decile[["weight", "inv_ret"]] * (
                -1
            )
            to_df = pd.concat([sell_decile, buy_decile])

            if i > 0:
                tto_df = pd.DataFrame(
                    index=np.unique(list(to_df.index) + list(prev_to_df.index))
                )
                try:
                    tto_df["cur_weight"] = to_df["weight"]
                except ValueError:
                    pdb.set_trace()
                tto_df[["prev_weight", "ret", "inv_ret"]] = prev_to_df[["weight", ret_name, "inv_ret"]]
                tto_df.fillna(0, inplace=True)
                denom = 1 + np.sum(tto_df["inv_ret"])
                turnover[i - 1] = np.sum(
                    (
                            tto_df["cur_weight"]
                            - tto_df["prev_weight"] * (1 + tto_df["ret"]) / denom
                    ).abs()
                )
                turnover[i - 1] = turnover[i - 1] * 0.5
            prev_to_df = to_df

        portfolio_ret = portfolio_ret.fillna(0)
        portfolio_ret["H-L"] = portfolio_ret[cut - 1] - portfolio_ret[0]
        """print(
            f"Spearman Corr between Prob and Stock Return is {np.nanmean(prob_ret_corr):.4f}"
        )
        print(
            f"Pearson Corr between Prob and Stock Return is {np.nanmean(prob_ret_pearson_corr):.4f}"
        )
        print(
            f"Spearman Corr between Prob and Top/Bottom deciles Return is {np.nanmean(prob_inv_ret_corr):.4f}"
        )
        print(
            f"Pearson Corr between Prob and Top/Bottom deciles Return is {np.nanmean(prob_inv_ret_pearson_corr):.4f}"
        )"""
        return portfolio_ret, np.mean(turnover)

    @staticmethod
    def _ret_to_cum_log_ret(rets):
        log_rets = np.log(rets.astype(float) + 1)
        return log_rets.cumsum()

    def make_portfolio_plot(
            self, portfolio_ret, cut, weight_type, save_path, plot_title
    ):
        ret_name = "nxt_freq_ewret" if weight_type == "ew" else "nxt_freq_vwret"
        df = portfolio_ret.copy()
        df.columns = ["Low(L)"] + [str(i) for i in range(2, cut)] + ["High(H)", "H-L"]
        spy = eqd.get_spy_freq_rets(self.freq)
        df["SPY"] = spy[ret_name]
        df.dropna(inplace=True)
        top_col_name, bottom_col_name = f"Top {cut}", f"Bottom {cut}"

        fig, ax = plt.subplots(figsize=(8, 6))
        df[[top_col_name, bottom_col_name, "SPY"]].cumsum().plot(ax=ax)
        ax.set_title(plot_title)
        plt.legend(loc="upper left")
        plt.savefig(save_path)
        plt.close()
        print(f"Portfolio plot saved to {save_path}")

    def portfolio_res_summary(self, portfolio_ret, turnover, cut=10):
        avg = portfolio_ret.mean().to_numpy()
        std = portfolio_ret.std().to_numpy()
        res = np.zeros((cut + 1, 3))
        period = 52 if self.freq == "week" else 12 if self.freq == "month" else 4
        res[:, 0] = avg * period
        res[:, 1] = std * np.sqrt(period)
        res[:, 2] = res[:, 0] / res[:, 1]

        summary_df = pd.DataFrame(res, columns=["ret", "std", "SR"])
        summary_df = summary_df.set_index(
            pd.Index(["Low"] + list(range(2, int(cut))) + ["High", "H-L"])
        )
        summary_df.loc["Turnover", "SR"] = turnover / (
            1 / 4
            if self.freq == "week"
            else 1
            if self.freq == "month"
            else 3
            if self.freq == "quarter"
            else 12
        )
        print(summary_df)
        return summary_df

    def generate_portfolio(
            self, weight_type, cut=10, delay=0, **kwargs
    ):
        assert weight_type in ["ew", "vw"]
        assert delay == 0  # 只支持 delay = 0
        pf_name = self.get_portfolio_name(weight_type, cut, delay)
        
        portfolio_ret, turnover = self.calculate_portfolio_rets(
            weight_type, cut, delay
        )

        data_dir = os.path.abspath(os.path.join(self.portfolio_dir, "pf_data"))
        print("portfolio_dir:", self.portfolio_dir)
        print("data_dir:", data_dir)

            # 确保文件夹存在
        os.makedirs(data_dir, exist_ok=True)

        csv_path = os.path.join(data_dir, f"pf_data_{pf_name}.csv")
        print("CSV file path:", csv_path)
        print("Portfolio Data:", portfolio_ret.head())  # 打印部分数据，确保数据格式正确

        try:
            portfolio_ret.to_csv(csv_path)
            print(f"CSV file saved successfully: {csv_path}")
        except Exception as e:
            print(f"Error saving CSV file: {e}")

        summary_df = self.portfolio_res_summary(portfolio_ret, turnover, cut)
        smry_path = os.path.join(self.portfolio_dir, f"{pf_name}.csv")
        print("smry_path:", smry_path)

            # 保存汇总文件
        try:
            summary_df.to_csv(smry_path)
            print(f"Summary saved to {smry_path}")
        except Exception as e:
            print(f"Error saving summary CSV file: {e}")




    def get_portfolio_name(self, weight_type, cut, delay):
        assert weight_type.lower() in ["ew", "vw"]
        delay_prefix = "" if delay == 0 else f"{delay}d_delay_"
        cut_surfix = "" if cut == 10 else f"_{cut}cut"
        custom_ret_surfix = "" if self.custom_ret is None else self.custom_ret
        tc_surfix = "" if not self.transaction_cost else "_w_transaction_cost"
        pf_name = f"{delay_prefix}{weight_type.lower()}{cut_surfix}{custom_ret_surfix}{tc_surfix}"
        return pf_name

    def load_portfolio_ret(self, weight_type, cut=10, delay=0):
        pf_name = self.get_portfolio_name(weight_type, cut, delay)
        try:
            df = pd.read_csv(
                op.join(self.portfolio_dir, "pf_data", f"pf_data_{pf_name}.csv"),
                index_col=0,
            )
        except FileNotFoundError:
            df = pd.read_csv(
                op.join(self.portfolio_dir, "pf_data", f"pf_data_{pf_name}_100.csv"),
                index_col=0,
            )
        df.index = pd.to_datetime(df.index)
        return df

    def load_portfolio_summary(self, weight_type, cut=10, delay=0):
        pf_name = self.get_portfolio_name(weight_type, cut, delay)
        try:
            df = pd.read_csv(op.join(self.portfolio_dir, f"{pf_name}.csv"), index_col=0)
        except FileNotFoundError:
            df = pd.read_csv(
                op.join(self.portfolio_dir, f"{pf_name}_100.csv"), index_col=0
            )
        return df


def main():

    # 创建 DataFrame
    signal_df = pd.DataFrame(pd.read_csv('data/resources/bagging_model_predictions.csv'))
    signal_df['ending_date'] = pd.to_datetime(signal_df['ending_date'], errors='coerce')
    signal_df.set_index(["ending_date", "StockID"], inplace=True)

    # 假设 portfolio_dir 是保存结果的文件夹路径
    portfolio_dir = 'outputs'

    # 创建 PortfolioManager 实例
    portfolio_manager = PortfolioManager(
        signal_df=signal_df,
        freq="month",  # 设置为月度重平衡
        portfolio_dir=portfolio_dir,
        start_year=1987,  # 分析开始年份
        end_year=2016,  # 分析结束年份
        delay_list=[0],  # 设置延迟为 0
        load_signal=True,  # 加载信号数据
        transaction_cost=False  # 不考虑交易成本
    )

    # 计算投资组合的回报（等权重）
    portfolio_ret = portfolio_manager.generate_portfolio(
        weight_type="ew",  # 使用等权重
        cut=10,  # 将股票分为 5 个分位（deciles）
        delay=0  # 延迟为 0
    )

    # 输出投资组合的回报
    print(portfolio_ret)

    # 获取投资组合的回报总结
    portfolio_summary = portfolio_manager.load_portfolio_summary(weight_type="ew", cut=10)
    print(portfolio_summary)


if __name__ == "__main__":
    main()

## NN1

In [ ]:
## NN1

import torch
import torch.nn as nn


# 定义 NN1 模型
class NN1(nn.Module):
    def __init__(self, input_dim, hidden_neurons=128):  # 添加 hidden_neurons 参数
        super(NN1, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_neurons)  # 使用 hidden_neurons 作为隐藏层大小
        self.out = nn.Linear(hidden_neurons, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.out(x)


model = NN1(input_dim, hidden_neurons=128)

# 多轮训练
for run_idx in range(1,num_runs + 1):
    print(f"Starting Run {run_idx + 1}/{num_runs}")
    # 每轮重新初始化DataLoader
    train_loader, test_loader = create_dataloaders(
        data_dir, start_year_train, start_year_test, end_year_test, task='regression'
    )
    # 每轮重新初始化模型
    model = NN2(input_dim, hidden_neurons=128)
    # 训练模型一次并保存
    model_path, final_r2, total_duration = train_model_once(
        model, train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim,
        num_epochs=50, lr=0.001, patience=10,
        save_base_path=base_save_path, run_idx=run_idx
    )
    print(f"Run {run_idx} completed. Model saved to: {model_path}. Final R²: {final_r2}. Duration: {total_duration:.2f} seconds.")
    gc.collect()

## NN2

In [ ]:
import torch
import torch.nn as nn

# 定义 NN2 模型
class NN2(nn.Module):
    def __init__(self, input_dim, hidden_neurons=128, dropout_rate=0.3):
        super(NN2, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_neurons)
        self.bn1 = nn.BatchNorm1d(hidden_neurons)
        self.fc2 = nn.Linear(hidden_neurons, hidden_neurons // 2)
        self.bn2 = nn.BatchNorm1d(hidden_neurons // 2)
        self.out = nn.Linear(hidden_neurons // 2, 1)
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        return self.out(x)


# 多轮训练
for run_idx in range(1,num_runs + 1):
    print(f"Starting Run {run_idx + 1}/{num_runs}")
    # 每轮重新初始化DataLoader
    train_loader, test_loader = create_dataloaders(
        data_dir, start_year_train, start_year_test, end_year_test, task='regression'
    )
    # 每轮重新初始化模型
    model = NN2(input_dim, hidden_neurons=128)
    # 训练模型一次并保存
    model_path, final_r2, total_duration = train_model_once(
        model, train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim,
        num_epochs=25, lr=0.03, patience=8,
        save_base_path=base_save_path, run_idx=run_idx
    )
    print(f"Run {run_idx} completed. Model saved to: {model_path}. Final R²: {final_r2}. Duration: {total_duration:.2f} seconds.")
    gc.collect()

## NN3

In [ ]:
# 定义 NN3 模型
class NN3(nn.Module):
    def __init__(self, input_dim, hidden_neurons1=128, hidden_neurons2=64, hidden_neurons3=32, dropout_rate1=0.3, dropout_rate2=0.4, dropout_rate3=0.5):
        super(NN3, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_neurons1)
        self.bn1 = nn.BatchNorm1d(hidden_neurons1)
        self.fc2 = nn.Linear(hidden_neurons1, hidden_neurons2)
        self.bn2 = nn.BatchNorm1d(hidden_neurons2)
        self.fc3 = nn.Linear(hidden_neurons2, hidden_neurons3)
        self.bn3 = nn.BatchNorm1d(hidden_neurons3)
        self.out = nn.Linear(hidden_neurons3, 1)
        self.dropout1 = nn.Dropout(dropout_rate1)
        self.dropout2 = nn.Dropout(dropout_rate2)
        self.dropout3 = nn.Dropout(dropout_rate3)
    
    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.dropout3(x)
        return self.out(x)


# 创建 NN3 模型实例

nn3_model = NN3(input_dim=input_dim, hidden_neurons1=128, hidden_neurons2=64, hidden_neurons3=32, dropout_rate1=0.3, dropout_rate2=0.4, dropout_rate3=0.5)

# 一次性训练模型并验证
model_path, final_r2, total_duration = train_model_once(
    nn3_model, train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim, 
    num_epochs=25, lr=0.03, patience=8, 
    save_base_path=base_save_path
)

print(f"训练完成。最佳模型保存在: {model_path}")
print(f"最终 R² 值: {final_r2}")
print(f"总训练时间: {total_duration:.2f} 秒")

gc.collect()

## 1D CNN

In [ ]:
import torch
import torch.nn as nn

# 定义 1D CNN 模型
class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes=1, dropout_rate=0.3):
        super(CNN1D, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.fc1 = nn.Linear(128 * (input_dim // 8), 64)  # 计算池化后的特征维度
        self.dropout = nn.Dropout(dropout_rate)
        self.out = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = x.unsqueeze(1)  # 添加通道维度，转换为 [batch_size, 1, input_dim]
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        x = torch.relu(self.bn3(self.conv3(x)))
        x = self.pool3(x)
        x = x.view(x.size(0), -1)  # 展平特征图
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.out(x)



# 创建 1D CNN 模型实例

model = CNN1D(input_dim=input_dim, dropout_rate=0.4)

# 调用train_model_once进行多次训练
def train_multiple_runs(model, train_loader, test_loader, input_dim, num_runs=5, num_epochs=25, lr=0.01, patience=8,
                        save_base_path=None):
    results = []
    for run_number in range(1, num_runs + 1):
        print(f"\nRunning model {run_number} of {num_runs}...")
        model_path, final_r2, total_duration = train_model_once(
            model, train_loader, test_loader, input_dim, num_epochs=num_epochs, lr=lr, patience=patience,
            save_base_path=save_base_path, run_number=run_number
        )
        results.append({
            'run_name': f"Run{run_number}",
            'model_path': model_path,
            'final_r2': final_r2,
            'total_duration': total_duration
        })

    return results


# 训练5次并保存结果
results = train_multiple_runs(
    model, train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim,
    num_runs=5, num_epochs=25, lr=0.01, patience=8, save_base_path=base_save_path
)

# 打印每次运行的结果
for result in results:
    if result['model_path']:
        print(f"Run: {result['run_name']}")
        print(f"  Best Model saved at: {result['model_path']}")
        print(f"  Final R²: {result['final_r2']}")
        print(f"  Total training time: {result['total_duration']:.2f} seconds\n")
    else:
        print(f"Run: {result['run_name']} did not produce a model.")

## rnn

In [ ]:
# 定义标准 RNN 模型
class StandardRNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout_rate=0.3):
        super(StandardRNNModel, self).__init__()
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout_rate, nonlinearity='tanh')
        self.fc = nn.Linear(hidden_dim, 64)
        self.dropout = nn.Dropout(dropout_rate)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        # 改变输入的形状为 (batch_size, seq_length=1, input_dim)
        x = x.unsqueeze(1)  # 添加一个时间维度 seq_length=1
        x, _ = self.rnn(x)  # 输出 x 的形状为 (batch_size, seq_length, hidden_dim)
        x = x[:, -1, :]  # 选择最后一个时间步的输出
        x = torch.relu(self.fc(x))
        x = self.dropout(x)
        return self.out(x)


# 创建标准 RNN 模型实例

model = StandardRNNModel(input_dim=input_dim, hidden_dim=64, num_layers=2, dropout_rate=0.3)



# 一次性训练模型并验证
model_path, final_r2, total_duration = train_model_once(
    model , train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim, 
    num_epochs=50, lr=0.03, patience=16, 
    save_base_path=base_save_path
)

print(f"训练完成。最佳模型保存在: {model_path}")
print(f"最终 R² 值: {final_r2}")
print(f"总训练时间: {total_duration:.2f} 秒")

gc.collect()

## LSTM

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout_rate=0.3):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout_rate)
        self.fc = nn.Linear(hidden_dim, 64)
        self.dropout = nn.Dropout(dropout_rate)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        x = x.unsqueeze(1)  # 添加一个时间维度 seq_length=1
        x, _ = self.lstm(x)  # x 形状为 (batch_size, seq_length, hidden_dim)
        x = x[:, -1, :]  # 选择最后一个时间步的输出
        x = torch.relu(self.fc(x))
        x = self.dropout(x)
        return self.out(x)

    
# 创建 RNN 模型实例

lstm_model = LSTMModel(input_dim=input_dim, hidden_dim=64, num_layers=2, dropout_rate=0.3)

# 一次性训练模型并验证
model_path, final_r2, total_duration = train_model_once(
    lstm_model , train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim, 
    num_epochs=50, lr=0.03, patience=16, 
    save_base_path=base_save_path,use_scheduler=True
)

print(f"训练完成。最佳模型保存在: {model_path}")
print(f"最终 R² 值: {final_r2}")
print(f"总训练时间: {total_duration:.2f} 秒")

gc.collect()

## Transformer

In [ ]:
import torch
import torch.nn as nn

# 定义简化版的 Transformer 模型
class SimpleTransformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=8, num_layers=2, dim_feedforward=256, dropout=0.1):
        super(SimpleTransformer, self).__init__()
        self.embedding = nn.Linear(input_dim, d_model)  # 输入嵌入层，将输入维度映射到 d_model
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.fc1 = nn.Linear(d_model, 64)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        # x 的形状为 (batch_size, input_dim) 或 (batch_size, seq_length, input_dim)
        if x.dim() == 2:
            x = x.unsqueeze(1)  # 增加一个时间维度，使得 x 的形状为 (batch_size, seq_length=1, input_dim)
            
        x = self.embedding(x)  # 将输入映射到 d_model 维度
        x = x.permute(1, 0, 2)  # 转换为 (seq_length, batch_size, d_model) 供 Transformer 使用
        x = self.transformer_encoder(x)
        x = x.permute(1, 0, 2)  # 转换回 (batch_size, seq_length, d_model)
        x = x[:, -1, :]  # 选择最后一个时间步的输出
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.out(x)

# 创建 Transformer 模型实例

transformer_model = SimpleTransformer(input_dim=input_dim, d_model=128, nhead=8, num_layers=2, dim_feedforward=256, dropout=0.1)



# 一次性训练模型并验证
model_path, final_r2, total_duration = train_model_once(
    transformer_model , train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim, 
    num_epochs=50, lr=0.005, patience=20, 
    save_base_path=base_save_path,use_scheduler=True
)

print(f"训练完成。最佳模型保存在: {model_path}")
print(f"最终 R² 值: {final_r2}")
print(f"总训练时间: {total_duration:.2f} 秒")

gc.collect()

## GNN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
import numpy as np
import os
import time
from tqdm import tqdm
import torch.optim as optim
import gc

# 定义简化版的 GNN 模型
class SimpleGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.3):
        super(SimpleGNN, self).__init__()
        
        # 定义图卷积层
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(input_dim, hidden_dim))  # 第一个卷积层，将 input_dim 映射到 hidden_dim
        
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))  # 中间层
        
        self.fc1 = nn.Linear(hidden_dim, 64)  # 全连接层
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(64, 1)  # 输出层

    def forward(self, x, edge_index, batch):
        # x 的形状为 (num_nodes, input_dim)
        # edge_index 表示边的连接关系，batch 表示节点所属的图
        
        # 依次通过图卷积层
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = self.dropout(x)
        
        # 对节点进行全局池化，得到每个图的图级表示
        x = global_mean_pool(x, batch)  # (batch_size, hidden_dim)
        
        # 全连接层
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        
        # 输出层
        return self.out(x)

# 创建 GNN 模型实例
#input_dim = 87  # 假设输入维度为 87


# 特供 GNN 的训练方法，改造输入数据为图数据

def transform_to_graph_data(X_batch, y_batch):
    batch_size = X_batch.size(0)
    num_nodes = X_batch.size(0)  # 每个样本被视为一个节点
    edge_index = torch.combinations(torch.arange(num_nodes), r=2).t().contiguous()  # 全连接图
    batch = torch.arange(batch_size, dtype=torch.long).repeat_interleave(1)  # 每个节点属于不同的图
    data = Data(x=X_batch, edge_index=edge_index, y=y_batch, batch=batch)
    return data


# 一次性训练模型并验证
def train_model_once_gnn(model, train_loader, test_loader, input_dim, num_epochs=50, lr=0.001, patience=10, save_base_path=None, use_scheduler=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Optionally add a learning rate scheduler
    scheduler = None
    if use_scheduler:
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    results = {'train_loss': [], 'test_loss': [], 'test_r2': [], 'learning_rate': []}

    # 创建模型文件夹
    model_folder = os.path.join(save_base_path, model.__class__.__name__)
    if not os.path.exists(model_folder):
        os.makedirs(model_folder)

    total_start_time = time.time()

    # 添加进度条来跟踪epoch进度
    epoch_iterator = tqdm(range(num_epochs), desc='Training Epochs', unit='epoch')
    for epoch in epoch_iterator:
        model.train()
        train_loss = 0.0

        # 在训练数据加载器上添加进度条
        train_loader_iter = tqdm(train_loader, desc=f"Epoch {epoch+1} Training", leave=False, unit='batch')
        for X_batch, y_batch in train_loader_iter:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            
            # 转换为图数据
            graph_data = transform_to_graph_data(X_batch, y_batch).to(device)
            
            # 前向传播
            outputs = model(graph_data.x, graph_data.edge_index, graph_data.batch)
            loss = criterion(outputs, graph_data.y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        
        train_loss /= len(train_loader.dataset)
        results['train_loss'].append(train_loss)

        model.eval()
        test_loss = 0.0
        y_test_pred_list = []
        y_test_list = []
        # 在测试数据加载器上添加进度条
        test_loader_iter = tqdm(test_loader, desc=f"Epoch {epoch+1} Testing", leave=False, unit='batch')
        with torch.no_grad():
            for X_batch, y_batch in test_loader_iter:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                
                # 转换为图数据
                graph_data = transform_to_graph_data(X_batch, y_batch).to(device)
                
                # 前向传播
                outputs = model(graph_data.x, graph_data.edge_index, graph_data.batch)
                loss = criterion(outputs, graph_data.y)
                test_loss += loss.item() * X_batch.size(0)
                y_test_pred_list.append(outputs.cpu().numpy())
                y_test_list.append(y_batch.cpu().numpy())
        
        test_loss /= len(test_loader.dataset)
        y_test_pred = np.concatenate(y_test_pred_list).ravel()
        y_test = np.concatenate(y_test_list).ravel()
        test_r2 = calculate_r2_oos(y_test, y_test_pred)
        
        results['test_loss'].append(test_loss)
        results['test_r2'].append(test_r2)

        # Log the current learning rate if scheduler is used
        if scheduler:
            current_lr = scheduler.optimizer.param_groups[0]['lr']
            results['learning_rate'].append(current_lr)

        # 更新进度条的描述信息
        epoch_iterator.set_postfix({
            'Train Loss': f"{train_loss:.6f}", 
            'Test Loss': f"{test_loss:.6f}", 
            'Test R²': f"{test_r2:.6f}",
            'LR': current_lr if use_scheduler else lr
        })

        # Use scheduler if applicable
        if scheduler:
            scheduler.step(test_loss)

        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            best_model_state = model.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    total_training_time = time.time() - total_start_time

    # 保存最佳模型
    if best_model_state:
        model.load_state_dict(best_model_state)
        model_save_path = os.path.join(model_folder, f"one_loop_best_model.pth")
        torch.save(model.state_dict(), model_save_path)

        # 保存训练历史
        history_save_path = os.path.join(model_folder, "one_loop_training_history.txt")
        with open(history_save_path, 'w') as f:
            f.write("Epoch\tTrain Loss\tTest Loss\tTest R²\tLR\n")
            for epoch in range(len(results['train_loss'])):
                f.write(f"{epoch+1}\t{results['train_loss'][epoch]:.6f}\t{results['test_loss'][epoch]:.6f}\t{results['test_r2'][epoch]:.6f}\t{results['learning_rate'][epoch] if use_scheduler else lr:.6f}\n")

        return model_save_path, results['test_r2'][-1], total_training_time
    else:
        return None, results['test_r2'][-1], total_training_time

gnn_model = SimpleGNN(input_dim=input_dim, hidden_dim=64, num_layers=2, dropout=0.3)


# 一次性训练模型并验证
model_path, final_r2, total_duration = train_model_once_gnn(
    gnn_model , train_loader_reg_one_loop, test_loader_reg_one_loop, input_dim, 
    num_epochs=50, lr=0.05, patience=30, 
    save_base_path=base_save_path,use_scheduler=True
)

print(f"训练完成。最佳模型保存在: {model_path}")
print(f"最终 R² 值: {final_r2}")
print(f"总训练时间: {total_duration:.2f} 秒")

gc.collect()